# 03. Transformations, Actions, and Schema Handling

This notebook focuses on the most common DataFrame operations in PySpark.

## Topics
- `select` and `withColumn`
- `filter` and `where`
- `drop`, `distinct`, `na` handling
- schema definition and casting
- null-safe expressions and conditional logic


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('DataFrameOps').master('local[*]').getOrCreate()

data = [
    (1, 'Alice', 28, 'HR'),
    (2, 'Bob', 35, 'IT'),
    (3, 'Charlie', 42, 'Data'),
    (4, None, 20, 'Sales')
]

df = spark.createDataFrame(data, ['id', 'name', 'age', 'department'])
df.show()

filtered = df.filter(F.col('age') >= 30)
filtered.show()

selected = df.select('id', 'name', F.col('age').alias('employee_age'))
selected.show()

with_bonus = df.withColumn('bonus', F.lit(500))
with_bonus.show()

spark.stop()


## Column expressions

Spark columns are expressions, not just Python values. This allows you to write readable and optimized transformations.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('ColumnOps').master('local[*]').getOrCreate()

df = spark.createDataFrame([(1, 100), (2, 250), (3, 400)], ['id', 'amount'])
df = df.withColumn('tax', F.col('amount') * 0.1)
df = df.withColumn('total', F.col('amount') + F.col('tax'))
df.show()

df = df.withColumn('amount_group', F.when(F.col('amount') > 200, 'high').otherwise('low'))
df.show()

spark.stop()


## Schema and casting


In [ ]:
from pyspark.sql import SparkSession, types as T

spark = SparkSession.builder.appName('SchemaDemo').master('local[*]').getOrCreate()

schema = T.StructType([
    T.StructField('id', T.IntegerType(), True),
    T.StructField('name', T.StringType(), True),
    T.StructField('salary', T.DoubleType(), True)
])

data = [(1, 'Alice', '2500'), (2, 'Bob', '3700')]
df = spark.createDataFrame(data, schema=['id', 'name', 'salary'])
df = df.withColumn('salary', df['salary'].cast('double'))
df.printSchema()
df.show()

spark.stop()


## Missing data handling


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('NullHandling').master('local[*]').getOrCreate()

df = spark.createDataFrame([(1, 'A'), (2, None), (None, 'C')], ['id', 'name'])
df.show()

print('Rows with null names:', df.filter(F.col('name').isNull()).count())
df = df.na.fill({'name': 'UNKNOWN'})
df.show()

spark.stop()


## Exercise
Build a DataFrame from a list of products and cast `price` to `DoubleType`, then add a discount column based on price thresholds.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('ProductExercise').master('local[*]').getOrCreate()

products = [
    (1, 'Laptop', '1200'),
    (2, 'Phone', '750'),
    (3, 'Tablet', '500')
]

df = spark.createDataFrame(products, ['id', 'product', 'price'])
df = df.withColumn('price', F.col('price').cast('double'))
df = df.withColumn('discount', F.when(F.col('price') >= 1000, 0.10).otherwise(0.05))
df.show()

spark.stop()
